# Database Loading and Verification Tutorial

This tutorial demonstrates how libskit manages atomic databases. When installed via `pip install`, databases are **bundled with the package** and loaded automatically on import — no manual setup required.

You can still override the database path if you want to use a custom database.

In [ ]:
import libskit

# The bundled atomic databases are activated automatically on import.
# Database.active() returns the handle to whichever Database is currently
# mirrored into libskit's C++ globals.
db = libskit.Database.active()
print(f"Active database: {db}")
print(f"Database path:   {db.path}")
print(f"Version:         {libskit.__version__}")


In [ ]:
# You can also construct an explicit handle to the bundled database and pass
# it via db= to model / Experiment constructors. When omitted, db= defaults
# to whichever Database is currently active (the bundled one, here).
db = libskit.Database.bundled()

## 1. Bundled Databases

When installed via pip, libskit ships with all required databases in the package. The databases are auto-loaded silently on `import libskit`. You can verify the current database path:

In [ ]:
# The bundled database lives inside the installed package data directory.
print(f"Bundled database path: {libskit.Database.bundled().path}")
print(f"Currently active:      {libskit.Database.active().path}")

# Verify it works right away (no manual loading needed).
model = libskit.LTEModel({"Fe": 1.0}, 10000.0, 1e23, 1e-3, db=db)
print(f"\nLTEModel created successfully: T={model.temperature} K")


## 2. Database Verification

Before using the databases, it's important to verify they contain valid data in the expected format.

In [ ]:
# Verify the databases
database_path = str(libskit.Database.active().path)
print("Verifying atomic databases...")
result = libskit.database_verification.verify_databases(database_path, sample_size=None, strict_mode=True)

# Print verification results
if result.is_valid:
    print("✅ All databases are valid!")
    print(f"Files found: {result.files_found}")
    
    # Show database statistics
    for filename, file_result in result.validation_results.items():
        if isinstance(file_result, dict) and 'total_records' in file_result:
            print(f"  {filename}: {file_result['total_records']} total records")
else:
    print("❌ Database verification failed:")
    for error in result.errors:
        print(f"  - {error}")


## 3. Using a Custom Database

The `Database` class is the supported way to point libskit at a different set of CSVs. Construct one of its variants and pass it via `db=` to every model / `Experiment` constructor (or call `db.activate()` to make it the process-wide default):

| Variant | Purpose |
| --- | --- |
| `libskit.Database.bundled()` | The CSVs that shipped with this wheel (default, auto-activated on import). |
| `libskit.Database.from_path(path)` | A directory of pre-processed CSVs you maintain yourself. |
| `libskit.Database.extended(...)` | Stark-matched lines plus every NIST line with imputed Stark FWHM. |
| `libskit.Database.filtered(top_n)` | Top-`top_n` strongest lines per ion across the extended database. |
| `libskit.Database.filtered_stark(top_n)` | Top-`top_n` strongest **Stark-matched** lines per ion (no imputation). |

The `db=` keyword is optional; when omitted it falls through to the currently-active database.


In [ ]:
# Example: load a custom database directory and use it explicitly.
# my_db = libskit.Database.from_path("/path/to/your/custom/databases")
# my_db.activate()
# model = libskit.LTEModel({"Fe": 1.0}, 10000.0, 1e23, 1e-3, db=my_db)

# Re-activate the bundled database with verbose output so we can see its
# contents being parsed. We force a reload by first activating a derived
# Database (filtered_stark with a tiny cutoff) and then switching back.
print("--- forcing a reload of the bundled database with verbose=2 ---")
libskit.Database.filtered_stark(top_n=1).activate()
libskit.Database.bundled().activate(verbose=2)